## Widgets

In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
CREATE widget TEXT catalog_name DEFAULT 'smartdata_project';
CREATE widget TEXT raw_name DEFAULT 'raw';
CREATE widget TEXT volume_name DEFAULT 'autoloader';
CREATE widget TEXT credential_name DEFAULT 'credential';
CREATE widget TEXT bronze_name DEFAULT 'bronze';
CREATE widget TEXT silver_name DEFAULT 'silver';
CREATE widget TEXT gold_name DEFAULT 'gold';

In [0]:
%python
catalog_name = dbutils.widgets.get('catalog_name')
raw_name = dbutils.widgets.get('raw_name')
volume_name = dbutils.widgets.get('volume_name')
credential_name = dbutils.widgets.get('credential_name')
bronze_name = dbutils.widgets.get('bronze_name')
silver_name = dbutils.widgets.get('silver_name')
gold_name = dbutils.widgets.get('gold_name')

## Catalogs

In [0]:
CREATE CATALOG IF NOT EXISTS ${catalog_name};

## Schemas

In [0]:
CREATE SCHEMA IF NOT EXISTS ${catalog_name}.${raw_name};
CREATE SCHEMA IF NOT EXISTS ${catalog_name}.${bronze_name};
CREATE SCHEMA IF NOT EXISTS ${catalog_name}.${silver_name};
CREATE SCHEMA IF NOT EXISTS ${catalog_name}.${gold_name};

## External Locations

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `extl-raw`
URL 'abfss://raw@adlssmartdataeastus2001.dfs.core.windows.net/sales'
WITH (STORAGE CREDENTIAL ${credential_name})
COMMENT 'External Location to the Raw Layer'

## Raw Layer

In [0]:
CREATE SCHEMA IF NOT EXISTS ${catalog_name}.${raw_name};
CREATE VOLUME IF NOT EXISTS ${catalog_name}.${raw_name}.${volume_name};

In [0]:
%python
import json
import os

watermarks_path = f'/Volumes/{catalog_name}/{raw_name}/autoloader/watermarks.json'

if not os.path.exists(watermarks_path):
    watermarks = [
        {"table": "sales",     "last_updated": "1900-01-01", "owner": "railgun@accelidol2gmail.onmicrosoft.com"},
        {"table": "persons",   "last_updated": "1900-01-01", "owner": "railgun@accelidol2gmail.onmicrosoft.com"},
        {"table": "products",  "last_updated": "1900-01-01", "owner": "railgun@accelidol2gmail.onmicrosoft.com"}
    ]
    with open(watermarks_path, "w") as f:
        json.dump(watermarks, f, indent=2)
    print("Watermarks inicializados.")
else:
    print("Watermarks ya existen, no se sobreescriben.")


## Bronze Layer

### Bronze Table dim_products

In [0]:
CREATE TABLE IF NOT EXISTS ${catalog_name}.${bronze_name}.bronze_products(
  product_id        VARCHAR(50),
  product_name      VARCHAR(255),
  price             FLOAT,
  category_name     VARCHAR(255),
  department_name   VARCHAR(255),
  created_at        TIMESTAMP,
  updated_at        TIMESTAMP,
  _ingestion_timestamp TIMESTAMP
);

In [0]:
CREATE TABLE IF NOT EXISTS ${catalog_name}.${bronze_name}.bronze_persons(
  customer_id_original  VARCHAR(50),
  full_name             VARCHAR(255),
  city                  VARCHAR(255),
  state                 VARCHAR(10),
  zipcode               BIGINT,
  rfm_id                BIGINT,
  age                   INT,
  updated_at            TIMESTAMP,
  created_at            TIMESTAMP,
  _ingestion_timestamp  TIMESTAMP
);

In [0]:
CREATE TABLE IF NOT EXISTS ${catalog_name}.${bronze_name}.bronze_sales(
  sales_id              VARCHAR(50),
  created_at            VARCHAR(50),
  end_at                VARCHAR(50),
  status                VARCHAR(50),
  product_id            VARCHAR(50),
  customer_id           VARCHAR(50),
  quantity              INT,
  sales_subtotal        FLOAT,
  updated_at            TIMESTAMP,
  medio_de_pago         VARCHAR(100),
  bank_origin           VARCHAR(100),
  _ingestion_timestamp  TIMESTAMP
);

## Silver Layer

In [0]:
CREATE TABLE IF NOT EXISTS ${catalog_name}.${silver_name}.silver_products(
  product_id        VARCHAR(50),
  product_name      VARCHAR(255),
  price             FLOAT,
  category_name     VARCHAR(255),
  department_name   VARCHAR(255),
  created_at        TIMESTAMP,
  updated_at        TIMESTAMP,
  _ingestion_timestamp TIMESTAMP
);

In [0]:
CREATE TABLE IF NOT EXISTS ${catalog_name}.${silver_name}.silver_persons(
  customer_id           VARCHAR(50),
  full_name             VARCHAR(255),
  city                  VARCHAR(255),
  state                 VARCHAR(10),
  zipcode               VARCHAR(50),
  rfm_id                VARCHAR(50),
  age                   INT,
  updated_at            TIMESTAMP,
  created_at            TIMESTAMP,
  _ingestion_timestamp  TIMESTAMP
);

In [0]:
CREATE TABLE IF NOT EXISTS ${catalog_name}.${silver_name}.silver_sales(
  sales_id              VARCHAR(50),
  created_at            TIMESTAMP,
  end_at                TIMESTAMP,
  status                VARCHAR(50),
  product_id            VARCHAR(50),
  customer_id           VARCHAR(50),
  quantity              INT,
  sales_subtotal        DECIMAL(10,2),
  updated_at            TIMESTAMP,
  medio_de_pago         VARCHAR(100),
  bank_origin           VARCHAR(100),
  _ingestion_timestamp  TIMESTAMP
);

## Gold Layer

In [0]:
CREATE TABLE IF NOT EXISTS ${catalog_name}.${gold_name}.gold_products (
    product_id            STRING,
    product_name          STRING,
    price                 DECIMAL(10, 2),
    category_name         STRING,
    department_name       STRING,
    created_at            TIMESTAMP,
    updated_at            TIMESTAMP,
    _ingestion_timestamp  TIMESTAMP,
    class_price           STRING NOT NULL
)

In [0]:
CREATE TABLE IF NOT EXISTS ${catalog_name}.${gold_name}.gold_persons (
    customer_id           STRING,
    full_name             STRING,
    city                  STRING,
    state                 STRING,
    zipcode               STRING,
    age                   INT,
    updated_at            TIMESTAMP,
    created_at            TIMESTAMP,
    _ingestion_timestamp  TIMESTAMP,
    class_age             STRING,
    rfm                   STRING
)

In [0]:
CREATE TABLE IF NOT EXISTS ${catalog_name}.${gold_name}.gold_sales (
    sales_id              STRING,
    created_at            TIMESTAMP,
    end_at                TIMESTAMP,
    status                STRING,
    product_id            STRING,
    customer_id           STRING,
    quantity              INT,
    sales_subtotal        DECIMAL(10, 2),
    updated_at            TIMESTAMP,
    medio_de_pago         STRING,
    bank_origin           STRING,
    _ingestion_timestamp  TIMESTAMP,
    class_quantity        STRING NOT NULL,
    class_salessub        STRING NOT NULL
)